# Forecasting with POIs and taxi flow

Compare the basic and spatial XGBoost feature sets, evaluate on 2025, and export the dataset used for neural experiments.

## Imports


In [ ]:
import pandas as pd
import numpy as np
import data_tools as data
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer


## Load demand and POI data


In [ ]:
df = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/yellow_taxi_2024_01_cleaned.parquet"
)
df_2025 = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/yellow_taxi_2025_features_neighbor_mean.parquet"
)


In [ ]:
counts = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/notebooks/zone_poi_counts_2023.parquet"
)
df_basic = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/yellow_taxi_2024_features.parquet"
)
df_spatial = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/yellow_taxi_2024_features_neighbor_mean.parquet"
)


## Add static POI features


In [ ]:
X_basic = df_basic.copy()
X_spatial = df_spatial.copy()
Test = df_2025.copy()


In [ ]:
poi_counts = counts.set_index("LocationID")
for X in [X_basic, X_spatial, Test]:
    X["restaurants"] = X["PULocationID"].map(poi_counts["restaurant_count"])
    X["hotels"] = X["PULocationID"].map(poi_counts["hotel_count"])


## Origin–destination connectivity and lagged flow


In [ ]:
# Connectivity matrix
P = (
    df.groupby(["PULocationID", "DOLocationID"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=range(1, 264), columns=range(1, 264), fill_value=0)
)
P_normalized = P.div(P.sum(axis=1).replace(0, np.nan), axis=0)


In [ ]:
for X in [X_spatial, X_basic, Test]:
    previous_pu = X.pivot(
        index="tpep_pickup_datetime", columns="PULocationID", values="lag_1h"
    ).reindex(columns=P_normalized.index)

    # Incoming flow: sum over source zones j of P[j, z] * pickups[j, t-1].
    flow = previous_pu @ P_normalized.fillna(0)
    X["flow"] = flow.to_numpy()[
        flow.index.get_indexer(X["tpep_pickup_datetime"]),
        flow.columns.get_indexer(X["PULocationID"]),
    ]


## Chronological development split


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, train, val, test = data.split(X_basic)

(
    X_train_spatial,
    X_val_spatial,
    X_test_spatial,
    y_train_spatial,
    y_val_spatial,
    y_test_spatial,
    train_spatial,
    val_spatial,
    test_spatial,
) = data.split(X_spatial)


In [ ]:
train_spatial


## Fit models and evaluate validation predictions


In [ ]:
model = data.model(train)
model_spatial = data.model(train_spatial)
model.fit(train, y_train)
model_spatial.fit(train_spatial, y_train_spatial)


In [ ]:
y_pred = model.predict(val)
y_pred_spatial = model_spatial.predict(val_spatial)


In [ ]:
data.metrics(y_val, y_pred)


In [ ]:
data.metrics(y_val_spatial, y_pred_spatial)


## Full-2024 refit and 2025 evaluation


In [ ]:
test = Test.drop(columns=["trip_count"])
y_test = Test["trip_count"]


In [ ]:
train = X_spatial.drop(columns=["tpep_pickup_datetime", "trip_count"])
y_train = X_spatial["trip_count"]


In [ ]:
model_spatial_2024 = data.model(train)
model_spatial_2024.fit(train, y_train)


In [ ]:
y_pred = model_spatial_2024.predict(test)


In [ ]:
data.metrics(y_test, y_pred)


## Export the deep learning dataset


In [ ]:
X_spatial.to_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/DL_Dataset.parquet",
    index=False,
)
